In [3]:
# ========================
# PARTE 0: INSTALAÇÕES (Google Colab)
# ========================
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium > /dev/null
!pip install pandas openpyxl > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGURAÇÕES
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, StaleElementReferenceException
import pandas as pd
import os, time, zipfile, requests
from datetime import datetime

options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# ========================
# PARTE 2: COLETAR LINKS DE PRODUTOS DE VÁRIAS PÁGINAS
# ========================
urls_origem = [
    "https://www.leroymerlin.com.br/search?term=porcelanato&searchTerm=porcelanato&searchType=default",
    "https://www.leroymerlin.com.br/search?term=Revestimentos%20de%20Parede"
]

todos_links = set()

for url in urls_origem:
    driver.get(url)
    pagina = 1
    print(f"\n🌐 Coletando da URL: {url}")

    while True:
        print(f"\U0001f504 Página {pagina}...")
        try:
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href*='/revestimento']")))
        except TimeoutException:
            print("❌ Timeout esperando os produtos.")
            break

        links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/revestimento']")
        for link in links:
            href = link.get_attribute("href")
            if href and "/revestimento" in href:
                todos_links.add(href)

        print(f"🔗 Total de links coletados até agora: {len(todos_links)}")

        try:
            botao = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button[rel='next']"))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", botao)
            time.sleep(1)
            botao.click()
            pagina += 1
            time.sleep(2)
        except (NoSuchElementException, TimeoutException, StaleElementReferenceException):
            print("⛔ Botão de próxima página não encontrado ou inválido.")
            break

print(f"\n✅ Total final de links coletados: {len(todos_links)}\n")

# ========================
# PARTE 3: EXTRAÇÃO DE DADOS
# ========================
def extrair_dados_produto(url):
    driver.get(url)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}
    dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text if driver.find_elements(By.CSS_SELECTOR, "h1") else None
    dados['Código'] = url.split("_")[-1] if "_" in url else None

    precos_candidatos = [
        '[data-cy="product-price"]',
        '[data-cy="product-price-integer"]',
        '[data-cy="product-price-sale"]',
        '.price__sales',
        '.value__integer'
    ]
    preco_txt = None
    for sel in precos_candidatos:
        elems = driver.find_elements(By.CSS_SELECTOR, sel)
        if elems:
            text = elems[0].text.strip()
            if text:
                preco_txt = text
                break

    if preco_txt:
        clean = preco_txt.replace("R$", "").replace("/m²", "").replace(".", "").replace(",", ".").strip()
        try:
            preco_val = float(clean)
            dados['Preço Promocional'] = f"{preco_val:.2f}".replace(".", ",")
        except:
            dados['Preço Promocional'] = "N/A"
    else:
        dados['Preço Promocional'] = "N/A"

    dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]') else None
    dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:image"]') else "N/A"

    specs = {}
    for tr in driver.find_elements(By.CSS_SELECTOR, 'table tr'):
        try:
            k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
            v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
            specs[k] = v
        except:
            continue
    dados['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items())
    dados['Loja'] = "Leroy Merlim"

    return dados

# ========================
# PARTE 4: COLETAR TODOS OS PRODUTOS
# ========================
todos = []
for i, link in enumerate(sorted(todos_links), 1):
    print(f"\U0001f50d {i}/{len(todos_links)} -> {link}")
    try:
        todos.append(extrair_dados_produto(link))
    except Exception as err:
        print("Erro:", err)
driver.quit()

# ========================
# PARTE 5: SALVAR PLANILHA E CSV COM ORDEM DEFINIDA
# ========================
data_nome = datetime.today().strftime("%Y-%m-%d")
pasta_base = "leroymerelin_coleta"
os.makedirs(pasta_base, exist_ok=True)

arquivo_excel = f"{pasta_base}/leroymerelin_{data_nome}.xlsx"
arquivo_csv = f"{pasta_base}/leroymerelin_{data_nome}.csv"

colunas_ordenadas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
df = pd.DataFrame(todos)
df.drop(columns=[c for c in df.columns if c in ['Erro']], inplace=True, errors='ignore')
df = df[colunas_ordenadas]
df.to_excel(arquivo_excel, index=False)
df.to_csv(arquivo_csv, index=False, sep=";")

print(f"📄 Excel salvo em: {arquivo_excel}")
print(f"📄 CSV salvo em: {arquivo_csv}")

# ========================
# PARTE 6: BAIXAR IMAGENS E ZIPAR
# ========================
pasta_imagens = os.path.join(pasta_base, f"imagens_leroymerelin_{data_nome}")
os.makedirs(pasta_imagens, exist_ok=True)

for _, r in df.iterrows():
    url_img = r['Imagem']
    cod = r['Código']
    if url_img and url_img != "N/A":
        for t in range(3):
            try:
                resp = requests.get(url_img, headers={"User-Agent":"Mozilla/5.0"}, timeout=10, stream=True)
                if resp.status_code == 200:
                    with open(f"{pasta_imagens}/{cod}.png", "wb") as f:
                        for cb in resp.iter_content(1024): f.write(cb)
                    break
            except:
                time.sleep(2)

zipname = os.path.join(pasta_base, f"imagens_leroymerelin_{data_nome}.zip")
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(pasta_imagens):
        z.write(os.path.join(pasta_imagens, fn), fn)
print(f"📦 Imagens zipadas em: {zipname}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file

🌐 Coletando da URL: https://www.leroymerlin.com.br/search?term=porcelanato&searchTerm=porcelanato&searchType=default
🔄 Página 1...
❌ Timeout esperando os produtos.

🌐 Coletando da URL: https://www.leroymerlin.com.br/search?term=Revestimentos%20de%20Parede
🔄 Página 1...
🔗 Total de links coletados até agora: 36


KeyboardInterrupt: 